In [1]:
# preprocess_and_build_graph_fixed_labels.py
"""
Preprocessing + TF-IDF + k-NN graph builder with corrected numeric label mapping.

Saves:
 - sampled_train_cleaned.csv
 - sampled_test_cleaned.csv (if test exists)
 - sampled_train_tfidf.csv
 - sampled_test_tfidf.csv (if test exists)
 - tfidf_vectorizer.joblib
 - label_encoder_mapping.joblib
 - graph_data_plain.pt  (plain torch dict: x, edge_index, y, train_mask, val_mask, test_mask)
"""

import os
import re
import string
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
import torch
import random
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ---------------------- CONFIG ----------------------
DATA_INPUT_DIR = "data"
TRAIN_CSV = os.path.join(DATA_INPUT_DIR, "train.csv")
TEST_CSV  = os.path.join(DATA_INPUT_DIR, "test.csv")

OUTPUT_DIR = "sampled_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sampling sizes per class (adjust)
N_PER_LABEL_TRAIN = 2000
N_PER_LABEL_TEST  = 1000

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

TFIDF_MAX_FEATURES = 5000
K_NEIGHBORS = 10

# Filenames
SAMP_TRAIN_CLEAN = os.path.join(OUTPUT_DIR, "sampled_train_cleaned.csv")
SAMP_TEST_CLEAN  = os.path.join(OUTPUT_DIR, "sampled_test_cleaned.csv")
TFIDF_TRAIN_CSV  = os.path.join(OUTPUT_DIR, "sampled_train_tfidf.csv")
TFIDF_TEST_CSV   = os.path.join(OUTPUT_DIR, "sampled_test_tfidf.csv")
VECTORIZER_PATH  = os.path.join(OUTPUT_DIR, "tfidf_vectorizer.joblib")
LABEL_MAP_PATH   = os.path.join(OUTPUT_DIR, "label_encoder_mapping.joblib")
GRAPH_PLAIN_PATH = os.path.join(OUTPUT_DIR, "graph_data_plain.pt")
# ----------------------------------------------------

# ---------- Utilities ----------
def find_label_column(df):
    """Try common label column names, otherwise pick a numeric-like column with small unique count."""
    candidates = ['label','Label','target','class','category','y']
    for c in candidates:
        if c in df.columns:
            return c
    # fallback: choose a column with numeric-like dtype or small unique count
    for c in df.columns:
        nunique = df[c].nunique()
        if 2 <= nunique <= 100 and df[c].dtype != object:
            return c
    # fallback: choose first object column that has small unique count
    for c in df.columns:
        nunique = df[c].nunique()
        if 2 <= nunique <= 100 and df[c].dtype == object:
            return c
    raise ValueError("Could not auto-detect a label column. Rename it to 'label' or 'target' or similar.")

# ensure nltk resources
def init_nltk():
    try:
        nltk.data.find('corpora/stopwords')
    except LookupError:
        nltk.download('stopwords')
    try:
        nltk.data.find('corpora/wordnet')
    except LookupError:
        nltk.download('wordnet')
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        nltk.download('punkt')

init_nltk()
STOPWORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    s = str(text)
    s = re.sub(r'[^\x00-\x7F]+',' ', s)  # remove non-ascii
    s = s.lower()
    s = s.translate(str.maketrans('', '', string.punctuation))
    s = re.sub(r'\d+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    tokens = [lemmatizer.lemmatize(tok) for tok in s.split() if tok and tok not in STOPWORDS]
    return ' '.join(tokens)

# ---------- Load CSVs ----------
if not os.path.exists(TRAIN_CSV):
    raise FileNotFoundError(f"Training CSV not found at {TRAIN_CSV}")
if not os.path.exists(TEST_CSV):
    warnings.warn(f"Test CSV not found at {TEST_CSV} — continuing without test sampling.")
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV) if os.path.exists(TEST_CSV) else None

# ---------- Detect label column ----------
label_col = find_label_column(train_df)
print("Detected label column:", label_col)

# keep a raw copy for debugging and consistent processing
train_df['label_raw'] = train_df[label_col].astype(str).str.strip()
if test_df is not None:
    test_df['label_raw'] = test_df[label_col].astype(str).str.strip()

# ---------- Stratified sampling ----------
def stratified_sample(df, label_raw_col, n_per_label, random_state=RANDOM_STATE):
    groups = df.groupby(label_raw_col, group_keys=False)
    sampled = []
    insufficient = {}
    for label, g in groups:
        count = len(g)
        if count >= n_per_label:
            sampled.append(g.sample(n=n_per_label, random_state=random_state))
        else:
            sampled.append(g)
            need = n_per_label - count
            if count == 0:
                insufficient[label] = 0
            else:
                up = g.sample(n=need, replace=True, random_state=random_state)
                sampled.append(up)
                insufficient[label] = count
    if len(sampled) == 0:
        return pd.DataFrame(columns=df.columns), insufficient
    result = pd.concat(sampled).reset_index(drop=True)
    return result, insufficient

print("Sampling train:", N_PER_LABEL_TRAIN, "per label")
train_sampled, train_insuff = stratified_sample(train_df, 'label_raw', N_PER_LABEL_TRAIN)
if train_insuff:
    print("Warning: some labels had fewer than requested and were upsampled:", train_insuff)

if test_df is not None:
    print("Sampling test:", N_PER_LABEL_TEST, "per label")
    test_sampled, test_insuff = stratified_sample(test_df, 'label_raw', N_PER_LABEL_TEST)
    if test_insuff:
        print("Warning: some test labels had fewer than requested and were upsampled:", test_insuff)
else:
    test_sampled = None

# ---------- Detect & combine text columns ----------
def detect_text_columns(df):
    names = [c.lower() for c in df.columns]
    if 'text' in names:
        # return the original column name matching 'text'
        return df.columns[names.index('text')]
    title = None; content = None
    for c in df.columns:
        lc = c.lower()
        if lc in ('title','headline','subject'):
            title = c
        if lc in ('content','body','article','description'):
            content = c
    if title and content:
        return (title, content)
    # fallback: first object/string column that is not 'label_raw' or similar
    for c in df.columns:
        if df[c].dtype == object and c not in ('label_raw', label_col):
            return c
    raise ValueError("Could not detect text columns. Ensure CSV contains title/content/text columns.")

def make_text_series(df):
    cols = detect_text_columns(df)
    if isinstance(cols, (tuple, list)):
        tcol, ccol = cols
        return (df[tcol].fillna('') + ' ' + df[ccol].fillna('')).astype(str)
    else:
        return df[cols].fillna('').astype(str)

print("Detecting text columns and cleaning text (this may take a while)...")
train_text_series = make_text_series(train_sampled)
train_sampled['text'] = train_text_series.apply(clean_text)

if test_sampled is not None:
    test_text_series = make_text_series(test_sampled)
    test_sampled['text'] = test_text_series.apply(clean_text)

train_sampled.to_csv(SAMP_TRAIN_CLEAN, index=False)
print("Saved cleaned train sample to:", SAMP_TRAIN_CLEAN)
if test_sampled is not None:
    test_sampled.to_csv(SAMP_TEST_CLEAN, index=False)
    print("Saved cleaned test sample to:", SAMP_TEST_CLEAN)

# ---------- Create corrected label mapping ----------
# Unique raw labels from the sampled training set:
unique_raw_labels = list(train_sampled['label_raw'].unique())
print("Unique raw labels (examples):", unique_raw_labels[:30])
# Check if all unique labels are integer-like (e.g., '0','1','10')
def all_int_like(vals):
    try:
        for v in vals:
            int(v)
        return True
    except Exception:
        return False

if all_int_like(unique_raw_labels):
    # sort numerically by integer value
    unique_sorted = sorted(unique_raw_labels, key=lambda x: int(x))
    print("Labels appear numeric-like — sorting numerically.")
else:
    unique_sorted = sorted(unique_raw_labels)
    print("Labels are non-numeric — sorting lexically.")

# Map original raw label string -> integer ID 0..C-1 in sorted order
label2int = {lab: idx for idx, lab in enumerate(unique_sorted)}
int2label = {v: k for k, v in label2int.items()}
print("Final label mapping (label_raw -> int):", label2int)

# Save mapping
joblib.dump({"label2int": label2int, "int2label": int2label}, LABEL_MAP_PATH)
print("Saved label mapping to:", LABEL_MAP_PATH)

# Apply mapping to sampled dataframes
train_sampled['label_encoded'] = train_sampled['label_raw'].map(label2int).astype(int)
if test_sampled is not None:
    # Map test labels; unseen test labels -> -1
    test_sampled['label_encoded'] = test_sampled['label_raw'].map(label2int)
    unseen = test_sampled['label_encoded'].isna().sum()
    if unseen > 0:
        print(f"Warning: {unseen} test rows have labels unseen in train. Setting to -1.")
        test_sampled['label_encoded'] = test_sampled['label_encoded'].fillna(-1).astype(int)

# ---------- TF-IDF fit on train.text ----------
print(f"Fitting TF-IDF (max_features={TFIDF_MAX_FEATURES}) on training text...")
vectorizer = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES)
X_train = vectorizer.fit_transform(train_sampled['text'].values)
joblib.dump(vectorizer, VECTORIZER_PATH)
print("Saved TF-IDF vectorizer to:", VECTORIZER_PATH)

X_train_arr = X_train.toarray().astype(np.float32)
tfidf_train_df = pd.DataFrame(X_train_arr, columns=vectorizer.get_feature_names_out())
tfidf_train_df['label'] = train_sampled['label_encoded'].values
tfidf_train_df.to_csv(TFIDF_TRAIN_CSV, index=False)
print("Saved train TF-IDF CSV to:", TFIDF_TRAIN_CSV)

if test_sampled is not None:
    X_test = vectorizer.transform(test_sampled['text'].values)
    X_test_arr = X_test.toarray().astype(np.float32)
    tfidf_test_df = pd.DataFrame(X_test_arr, columns=vectorizer.get_feature_names_out())
    if 'label_raw' in test_sampled.columns:
        tfidf_test_df['label_raw'] = test_sampled['label_raw'].values
    tfidf_test_df['label_encoded'] = test_sampled['label_encoded'].values
    tfidf_test_df.to_csv(TFIDF_TEST_CSV, index=False)
    print("Saved test TF-IDF CSV to:", TFIDF_TEST_CSV)

# ---------- Diagnostics ----------
n_classes = len(label2int)
print("Number of classes in training (n_classes):", n_classes)
if n_classes <= 1:
    print("ERROR: <=1 class detected. Dumping label details for debugging:")
    print(train_sampled[['label_raw','label_encoded']].head(50))
    raise ValueError("Training contains <=1 class. Check input labels and detection.")

# ---------- Build k-NN graph (cosine) from training TF-IDF features ----------
print(f"Building k-NN graph (K={K_NEIGHBORS}) from training features...")
knn = NearestNeighbors(n_neighbors=K_NEIGHBORS+1, metric='cosine', n_jobs=-1)
knn.fit(X_train_arr)
_, indices = knn.kneighbors(X_train_arr, return_distance=True)

edge_u = []
edge_v = []
n_nodes = X_train_arr.shape[0]
for i in range(n_nodes):
    neighs = indices[i, 1:]  # skip self
    for j in neighs:
        edge_u.append(i); edge_v.append(int(j))
        edge_u.append(int(j)); edge_v.append(i)

edge_index = np.vstack([edge_u, edge_v]).astype(np.int64)

# dedupe
pairs = set()
u_unique = []
v_unique = []
for a,b in zip(edge_index[0], edge_index[1]):
    if (a,b) not in pairs:
        pairs.add((a,b))
        u_unique.append(a); v_unique.append(b)
edge_index = np.vstack([u_unique, v_unique]).astype(np.int64)
print("Num nodes:", n_nodes, "Num edges:", edge_index.shape[1])

# ---------- Create stratified masks within sampled training set ----------
all_idx = np.arange(n_nodes)
train_idx, testval_idx, y_train_enc, y_testval_enc = train_test_split(
    all_idx, train_sampled['label_encoded'].values, test_size=0.30,
    stratify=train_sampled['label_encoded'].values, random_state=RANDOM_STATE
)
val_idx, test_idx, _, _ = train_test_split(testval_idx, y_testval_enc, test_size=0.5,
                                           stratify=y_testval_enc, random_state=RANDOM_STATE)

train_mask = torch.zeros(n_nodes, dtype=torch.bool); train_mask[train_idx] = True
val_mask = torch.zeros(n_nodes, dtype=torch.bool); val_mask[val_idx] = True
test_mask = torch.zeros(n_nodes, dtype=torch.bool); test_mask[test_idx] = True

# ---------- Save plain graph dict ----------
plain = {
    "x": torch.from_numpy(X_train_arr),                    # [n_nodes, n_features]
    "edge_index": torch.from_numpy(edge_index),            # [2, n_edges]
    "y": torch.from_numpy(train_sampled['label_encoded'].values).long(),
    "train_mask": train_mask,
    "val_mask": val_mask,
    "test_mask": test_mask
}
torch.save(plain, GRAPH_PLAIN_PATH)
print("Saved plain graph data to:", GRAPH_PLAIN_PATH)

# Save cleaned sampled CSVs again with label_encoded (already saved earlier but update)
train_sampled.to_csv(SAMP_TRAIN_CLEAN, index=False)
print("Saved final cleaned sampled train CSV to:", SAMP_TRAIN_CLEAN)
if test_sampled is not None:
    test_sampled.to_csv(SAMP_TEST_CLEAN, index=False)
    print("Saved final cleaned sampled test CSV to:", SAMP_TEST_CLEAN)

print("Preprocessing pipeline finished successfully.")


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\himas\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Detected label column: label
Sampling train: 2000 per label
Sampling test: 1000 per label
Detecting text columns and cleaning text (this may take a while)...
Saved cleaned train sample to: sampled_data\sampled_train_cleaned.csv
Saved cleaned test sample to: sampled_data\sampled_test_cleaned.csv
Unique raw labels (examples): ['0', '1', '10', '11', '12', '13', '2', '3', '4', '5', '6', '7', '8', '9']
Labels appear numeric-like — sorting numerically.
Final label mapping (label_raw -> int): {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, '10': 10, '11': 11, '12': 12, '13': 13}
Saved label mapping to: sampled_data\label_encoder_mapping.joblib
Fitting TF-IDF (max_features=5000) on training text...
Saved TF-IDF vectorizer to: sampled_data\tfidf_vectorizer.joblib
Saved train TF-IDF CSV to: sampled_data\sampled_train_tfidf.csv
Saved test TF-IDF CSV to: sampled_data\sampled_test_tfidf.csv
Number of classes in training (n_classes): 14
Building k-NN graph (K=10) fro

In [2]:
# 2_graphsage_train_from_plain.py
import os
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv
import numpy as np

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_DIR = "sampled_data"
PLAIN_PATH = os.path.join(DATA_DIR, "graph_data_plain.pt")
MODEL_PATH = os.path.join(DATA_DIR, "graphsage_model.pth")
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(PLAIN_PATH):
    raise FileNotFoundError(f"{PLAIN_PATH} not found. Run graph construction first.")

# Load the plain dict
d = torch.load(PLAIN_PATH)
x = d["x"]
edge_index = d["edge_index"]
y = d["y"]
train_mask = d["train_mask"]
val_mask = d["val_mask"]
test_mask = d["test_mask"]

# Build PyG Data
data = Data(x=x, edge_index=edge_index, y=y)
data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)

in_channels = data.num_node_features
num_classes = int(data.y.max().item() + 1)

# Model
class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=2, dropout=0.5):
        super().__init__()
        self.num_layers = num_layers
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
        if num_layers > 1:
            self.convs.append(SAGEConv(hidden_channels, out_channels))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i != len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return x

# Hyperparameters (tweak as needed)
HIDDEN = 256
NUM_LAYERS = 2
LR = 0.001
WEIGHT_DECAY = 5e-4
EPOCHS = 30
BATCH_SIZE = 512
NEIGHBORS = [15, 10]  # neighbors per layer for NeighborLoader

model = GraphSAGE(in_channels, HIDDEN, num_classes, num_layers=NUM_LAYERS, dropout=0.5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = torch.nn.CrossEntropyLoss()

train_idx = data.train_mask.nonzero(as_tuple=False).view(-1).tolist()
train_loader = NeighborLoader(
    data,
    num_neighbors=NEIGHBORS,
    input_nodes=train_idx,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

def evaluate(model, data, mask):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        preds = out.argmax(dim=1)
        acc = (preds[mask] == data.y[mask]).sum().item() / mask.sum().item()
        return acc

best_val = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_examples = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        target_n = batch.batch_size  # nodes with labels in this mini-batch
        loss = criterion(out[:target_n], batch.y[:target_n])
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * target_n
        total_examples += target_n
    avg_loss = total_loss / (total_examples + 1e-12)
    val_acc = evaluate(model, data, data.val_mask)
    test_acc = evaluate(model, data, data.test_mask)
    print(f"Epoch {epoch:03d} | Loss {avg_loss:.4f} | Val {val_acc:.4f} | Test {test_acc:.4f}")
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Saved best model (val {best_val:.4f}) to {MODEL_PATH}")

print("Training complete. Best val:", best_val)


Epoch 001 | Loss 2.1041 | Val 0.8686 | Test 0.8683
Saved best model (val 0.8686) to sampled_data\graphsage_model.pth
Epoch 002 | Loss 0.7454 | Val 0.8964 | Test 0.9010
Saved best model (val 0.8964) to sampled_data\graphsage_model.pth
Epoch 003 | Loss 0.4201 | Val 0.9086 | Test 0.9110
Saved best model (val 0.9086) to sampled_data\graphsage_model.pth
Epoch 004 | Loss 0.3432 | Val 0.9148 | Test 0.9224
Saved best model (val 0.9148) to sampled_data\graphsage_model.pth
Epoch 005 | Loss 0.3009 | Val 0.9262 | Test 0.9269
Saved best model (val 0.9262) to sampled_data\graphsage_model.pth
Epoch 006 | Loss 0.2689 | Val 0.9300 | Test 0.9314
Saved best model (val 0.9300) to sampled_data\graphsage_model.pth
Epoch 007 | Loss 0.2478 | Val 0.9336 | Test 0.9340
Saved best model (val 0.9336) to sampled_data\graphsage_model.pth
Epoch 008 | Loss 0.2294 | Val 0.9383 | Test 0.9383
Saved best model (val 0.9383) to sampled_data\graphsage_model.pth
Epoch 009 | Loss 0.2144 | Val 0.9398 | Test 0.9402
Saved best mo

In [5]:
# inference_memory_efficient_fixed.py
"""
Memory-efficient inference for GraphSAGE using NeighborLoader sampling.

Fixes previous AttributeError when temp Data objects don't contain `y`.
Outputs:
 - sampled_data/heldout_test_predictions_nb.csv
 - sampled_data/external_test_inductive_predictions_nb.csv (if external test found)
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv
from sklearn.neighbors import NearestNeighbors
import joblib
import gc

DATA_DIR = "sampled_data"
PLAIN_PATH = os.path.join(DATA_DIR, "graph_data_plain.pt")
MODEL_PATH = os.path.join(DATA_DIR, "graphsage_model.pth")
VECTORIZER_PATH = os.path.join(DATA_DIR, "tfidf_vectorizer.joblib")
TEST_TFIDF_CSV = os.path.join(DATA_DIR, "sampled_test_tfidf.csv")
TEST_CLEANED_CSV = os.path.join(DATA_DIR, "sampled_test_cleaned.csv")

# Tune these to reduce memory
NEIGHBORS_PER_LAYER = [15, 10]
BATCH_SIZE_TARGETS = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FORCE_CPU = False
if FORCE_CPU:
    DEVICE = torch.device("cpu")

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=2, dropout=0.5):
        super().__init__()
        self.num_layers = num_layers
        self.convs = torch.nn.ModuleList()
        self.convs.append(SAGEConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
        if num_layers > 1:
            self.convs.append(SAGEConv(hidden_channels, out_channels))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i != len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return x

def try_clear_cuda():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

def load_plain_graph(path):
    d = torch.load(path, map_location='cpu')
    x = d['x']
    edge_index = d['edge_index']
    y = d.get('y', None)
    train_mask = d.get('train_mask', None)
    val_mask = d.get('val_mask', None)
    test_mask = d.get('test_mask', None)
    data = Data(x=x, edge_index=edge_index, y=y)
    data.train_mask = train_mask
    data.val_mask = val_mask
    data.test_mask = test_mask
    return data

def load_model(in_channels, hidden, num_classes, num_layers, path, device):
    model = GraphSAGE(in_channels, hidden, num_classes, num_layers=num_layers, dropout=0.5)
    state = torch.load(path, map_location='cpu')
    model.load_state_dict(state)
    model = model.to(device)
    model.eval()
    return model

def neighbor_batch_predict(model, data, target_node_idx, neighbors_per_layer, batch_size, device):
    """
    Predict logits for nodes in `target_node_idx` using NeighborLoader batches.
    `data` can be a Data object with or without .y.
    Returns preds and probs aligned with target_node_idx order.
    """
    # Build a CPU Data containing only fields that exist to avoid AttributeError
    kwargs = {'x': data.x.cpu(), 'edge_index': data.edge_index.cpu()}
    if hasattr(data, 'y') and data.y is not None:
        kwargs['y'] = data.y.cpu()
    data_cpu = Data(**kwargs)

    loader = NeighborLoader(
        data_cpu,
        input_nodes=target_node_idx,
        num_neighbors=neighbors_per_layer,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    all_preds = []
    all_probs = []
    all_indices = []

    for batch in loader:
        batch = batch.to(device)
        with torch.no_grad():
            out = model(batch.x, batch.edge_index)
        # For NeighborLoader, the first batch.batch_size rows correspond to the target nodes
        target_n = int(batch.batch_size)
        logits_targets = out[:target_n].cpu()
        probs_targets = F.softmax(logits_targets, dim=1).numpy()
        preds_targets = logits_targets.argmax(dim=1).numpy()

        # get global node ids for these target nodes; batch.n_id exists in PyG NeighborLoader
        if hasattr(batch, 'n_id'):
            global_target_ids = batch.n_id[:target_n].cpu().numpy()
        else:
            # fallback: if n_id not present (unlikely), assume ordering matches requested slice
            # This is less safe, but keeps code robust.
            global_target_ids = np.array(target_node_idx[:target_n])

        all_preds.append(preds_targets)
        all_probs.append(probs_targets)
        all_indices.append(global_target_ids)

        # free memory
        del batch, out, logits_targets
        try_clear_cuda()

    if len(all_indices) == 0:
        return np.array([], dtype=int), np.zeros((0, model.convs[-1].out_features), dtype=float)

    indices_concat = np.concatenate(all_indices, axis=0)
    preds_concat = np.concatenate(all_preds, axis=0)
    probs_concat = np.concatenate(all_probs, axis=0)

    # Map global indices -> position
    idx_to_pos = {int(idx): i for i, idx in enumerate(indices_concat)}
    preds_out = np.zeros((len(target_node_idx),), dtype=int)
    probs_out = np.zeros((len(target_node_idx), probs_concat.shape[1]), dtype=float)
    for out_pos, gidx in enumerate(target_node_idx):
        pos = idx_to_pos.get(int(gidx), None)
        if pos is None:
            preds_out[out_pos] = -1
            probs_out[out_pos] = np.zeros((probs_concat.shape[1],), dtype=float)
        else:
            preds_out[out_pos] = preds_concat[pos]
            probs_out[out_pos] = probs_concat[pos]
    return preds_out, probs_out

def predict_heldout(data, model, device, neighbors_per_layer, batch_size):
    if data.test_mask is None:
        print("No test mask found. Skipping heldout.")
        return
    test_idx = data.test_mask.nonzero(as_tuple=False).view(-1).cpu().numpy()
    print(f"Held-out test nodes: {len(test_idx)}. Running neighbor-sampled inference...")
    preds, probs = neighbor_batch_predict(model, data, test_idx, neighbors_per_layer, batch_size, device)
    top3 = np.argsort(-probs, axis=1)[:, :3] if probs.size else np.zeros((len(preds),3), dtype=int)
    df = pd.DataFrame({
        "node_idx": test_idx,
        "pred_label": preds
    })
    if probs.size:
        df["top1"] = top3[:,0]; df["top2"] = top3[:,1]; df["top3"] = top3[:,2]
    out_path = os.path.join(DATA_DIR, "heldout_test_predictions_nb.csv")
    df.to_csv(out_path, index=False)
    print("Saved heldout predictions ->", out_path)

def predict_external_inductive(data, model, device, neighbors_per_layer, batch_size):
    if not os.path.exists(TEST_TFIDF_CSV) and not os.path.exists(TEST_CLEANED_CSV):
        print("No external test CSVs found. Skipping inductive external inference.")
        return

    # Load TF-IDF or recompute using vectorizer
    if os.path.exists(TEST_TFIDF_CSV):
        test_df = pd.read_csv(TEST_TFIDF_CSV)
        if any(c in test_df.columns for c in ['text','title_clean','content_clean']):
            vec = joblib.load(VECTORIZER_PATH)
            if 'text' in test_df.columns:
                text = test_df['text'].astype(str).values
            else:
                title = test_df.get('title_clean','').astype(str) if 'title_clean' in test_df.columns else ''
                content = test_df.get('content_clean','').astype(str) if 'content_clean' in test_df.columns else ''
                text = (title + ' ' + content).astype(str).values
            X_test = vec.transform(text).toarray().astype(np.float32)
        else:
            if 'label' in test_df.columns:
                test_df = test_df.drop(columns=['label'])
            X_test = test_df.values.astype(np.float32)
    else:
        vec = joblib.load(VECTORIZER_PATH)
        cleaned = pd.read_csv(TEST_CLEANED_CSV)
        if 'text' in cleaned.columns:
            text = cleaned['text'].astype(str).values
        else:
            title = cleaned.get('title_clean','').astype(str) if 'title_clean' in cleaned.columns else ''
            content = cleaned.get('content_clean','').astype(str) if 'content_clean' in cleaned.columns else ''
            text = (title + ' ' + content).astype(str).values
        X_test = vec.transform(text).toarray().astype(np.float32)

    n_new = X_test.shape[0]
    print(f"External test nodes: {n_new}")

    existing_feat = data.x.cpu().numpy()
    knn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric='cosine', n_jobs=-1)
    knn.fit(existing_feat)
    _, neighbors = knn.kneighbors(X_test, return_distance=True)

    n_old = existing_feat.shape[0]
    new_edges_u = []; new_edges_v = []
    for new_i, neighs in enumerate(neighbors):
        new_idx = n_old + new_i
        for nb in neighs:
            new_edges_u.append(new_idx); new_edges_v.append(int(nb))
            new_edges_u.append(int(nb)); new_edges_v.append(new_idx)

    old_edge = data.edge_index.cpu().numpy()
    all_u = np.concatenate([old_edge[0], np.array(new_edges_u, dtype=int)])
    all_v = np.concatenate([old_edge[1], np.array(new_edges_v, dtype=int)])
    new_edge_index = np.vstack([all_u, all_v])
    # dedupe
    pairs = set(); u_unique=[]; v_unique=[]
    for a,b in zip(new_edge_index[0], new_edge_index[1]):
        if (a,b) not in pairs:
            pairs.add((a,b)); u_unique.append(a); v_unique.append(b)
    new_edge_index = np.vstack([u_unique, v_unique]).astype(np.int64)

    X_all = np.vstack([existing_feat, X_test]).astype(np.float32)
    temp_data = Data(x=torch.from_numpy(X_all), edge_index=torch.from_numpy(new_edge_index))
    new_node_indices = np.arange(n_old, n_old + n_new)

    print("Running neighbor-sampled inference for new nodes (inductive).")
    preds_new, probs_new = neighbor_batch_predict(model, temp_data, new_node_indices, neighbors_per_layer, batch_size, device)

    top3 = np.argsort(-probs_new, axis=1)[:, :3] if probs_new.size else np.zeros((len(preds_new),3), dtype=int)
    out_df = pd.DataFrame({
        "new_node_idx": new_node_indices,
        "pred_label": preds_new
    })
    if probs_new.size:
        out_df["top1"] = top3[:,0]; out_df["top2"] = top3[:,1]; out_df["top3"] = top3[:,2]
    out_path = os.path.join(DATA_DIR, "external_test_inductive_predictions_nb.csv")
    out_df.to_csv(out_path, index=False)
    print("Saved inductive predictions ->", out_path)

def main():
    if not os.path.exists(PLAIN_PATH):
        raise FileNotFoundError(f"Graph plain file not found at {PLAIN_PATH}. Run preprocessing first.")
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"Trained model not found at {MODEL_PATH}. Run training first.")

    try_clear_cuda()
    data = load_plain_graph(PLAIN_PATH)
    in_channels = data.x.shape[1]
    num_classes = int(data.y.max().item() + 1) if (hasattr(data, 'y') and data.y is not None) else None
    if num_classes is None:
        raise ValueError("Graph data has no label information (y). Cannot infer num_classes.")

    HIDDEN = 256
    NUM_LAYERS = 2

    model = load_model(in_channels, HIDDEN, num_classes, NUM_LAYERS, MODEL_PATH, DEVICE)

    predict_heldout(data, model, DEVICE, NEIGHBORS_PER_LAYER, BATCH_SIZE_TARGETS)
    predict_external_inductive(data, model, DEVICE, NEIGHBORS_PER_LAYER, BATCH_SIZE_TARGETS)

    print("Inference done.")

if __name__ == "__main__":
    main()


Held-out test nodes: 4200. Running neighbor-sampled inference...
Saved heldout predictions -> sampled_data\heldout_test_predictions_nb.csv
External test nodes: 14000
Running neighbor-sampled inference for new nodes (inductive).
Saved inductive predictions -> sampled_data\external_test_inductive_predictions_nb.csv
Inference done.


In [ ]:
# predict_single_abstract_dbpedia.py
"""
Predict DBpedia-14 class name for a single new abstract.

Required files (in sampled_data/):
 - graph_data_plain.pt           (dict with keys "x" (tensor NxD) and "edge_index" (2xE tensor))
 - tfidf_vectorizer.joblib       (scikit-learn vectorizer used at training)
 - graphsage_model.pth           (PyTorch model state_dict saved after training)

Optional:
 - label_encoder_mapping.joblib  (dict or list mapping class indices -> labels)

This script:
 - loads vectorizer + plain graph data
 - converts input text to TF-IDF
 - attaches a new node by finding k nearest neighbors in TF-IDF space
 - builds a small subgraph (k-hop neighbor sampled) for inference
 - runs the trained GraphSAGE model on that subgraph
 - prints predicted class id, name and softmax probabilities
"""

import os
import sys
import joblib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from collections import defaultdict

# ---------- User settings ----------
DATA_DIR = "sampled_data"
PLAIN_PATH = os.path.join(DATA_DIR, "graph_data_plain.pt")
MODEL_PATH = os.path.join(DATA_DIR, "graphsage_model.pth")
VECTORIZER_PATH = os.path.join(DATA_DIR, "tfidf_vectorizer.joblib")
LABEL_MAP_PATH = os.path.join(DATA_DIR, "label_encoder_mapping.joblib")  # optional

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DBPEDIA_14_NAMES = [
    "Company", "EducationalInstitution", "Artist", "Athlete", "OfficeHolder",
    "MeanOfTransportation", "Building", "NaturalPlace", "Village", "Animal",
    "Plant", "Album", "Film", "WrittenWork"
]

# example input text you want to classify (edit as needed)
INPUT_TEXT = """
gaia river tributary ul mare river romania
"""

# GraphSAGE hyperparameters used at training: adapt if different
HIDDEN = 256
NUM_LAYERS = 2
DROPOUT = 0.5

# When connecting new node to existing graph, how many TF-IDF neighbors to attach
K_TFIDF = 10

# When building subgraph for inference, neighbors per layer (match training neighbor sizes if possible)
NEIGHBORS_PER_LAYER = [15, 10]  # example for a 2-layer model
# -----------------------------------

# ----- Helper: GraphSAGE model class (must match training) -----
class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=2, dropout=0.5):
        super().__init__()
        self.num_layers = num_layers
        self.convs = nn.ModuleList()
        if num_layers == 1:
            self.convs.append(SAGEConv(in_channels, out_channels))
        else:
            # in -> hidden
            self.convs.append(SAGEConv(in_channels, hidden_channels))
            # hidden -> hidden (num_layers - 2 times)
            for _ in range(num_layers - 2):
                self.convs.append(SAGEConv(hidden_channels, hidden_channels))
            # hidden -> out
            self.convs.append(SAGEConv(hidden_channels, out_channels))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            is_last = (i == len(self.convs) - 1)
            if not is_last:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return x

# ----- Helper: build adjacency list from edge_index (numpy) -----
def build_adj_list(edge_index_np, num_nodes=None):
    adj = defaultdict(list)
    src = edge_index_np[0]
    dst = edge_index_np[1]
    for u, v in zip(src, dst):
        adj[int(u)].append(int(v))
    if num_nodes is not None:
        # ensure every node has an entry (even if empty)
        for i in range(num_nodes):
            _ = adj[i]
    return adj

# ----- Helper: k-hop neighbor sampling to build a small induced subgraph -----
def build_subgraph_nodes(target_nodes, adj, neighbors_per_layer, seed=None):
    """
    Expand neighborhood around target_nodes layer by layer.
    For each node at each layer, sample up to neighbors_per_layer[layer] neighbors.
    Returns list of nodes in the induced subgraph (unique), and mapping dict old->new indices.
    """
    if seed is not None:
        np.random.seed(seed)
    layers = []
    current = set(int(x) for x in target_nodes)
    layers.append(current)
    all_nodes = set(current)
    for layer_k in neighbors_per_layer:
        next_nodes = set()
        for node in current:
            neighs = adj.get(node, [])
            if len(neighs) == 0:
                continue
            # sample up to layer_k neighbors (no replacement if possible)
            if len(neighs) <= layer_k:
                chosen = neighs
            else:
                chosen = list(np.random.choice(neighs, size=layer_k, replace=False))
            for n in chosen:
                next_nodes.add(int(n))
        if not next_nodes:
            break
        layers.append(next_nodes)
        all_nodes.update(next_nodes)
        current = next_nodes
    # return deterministic order (sorted) for stable indexing
    nodes_sorted = sorted(all_nodes)
    old_to_new = {old: new for new, old in enumerate(nodes_sorted)}
    return nodes_sorted, old_to_new

# ----- Helper: create subgraph edge_index and x tensor based on node list -----
def induced_subgraph(node_list, old_to_new, edge_index_np, x_all_np):
    """
    node_list: list of original node indices included
    old_to_new: mapping original->new index (0..n_sub-1)
    Returns: x_sub (torch.FloatTensor NxD), edge_index_sub (torch.LongTensor 2xE)
    """
    node_set = set(node_list)
    src = edge_index_np[0]
    dst = edge_index_np[1]
    u_sub = []
    v_sub = []
    for u, v in zip(src, dst):
        if (int(u) in node_set) and (int(v) in node_set):
            u_sub.append(old_to_new[int(u)])
            v_sub.append(old_to_new[int(v)])
    if len(u_sub) == 0:
        edge_index_sub = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index_sub = torch.tensor([u_sub, v_sub], dtype=torch.long)
    x_sub = torch.from_numpy(x_all_np[np.array(node_list, dtype=int)])
    return x_sub, edge_index_sub

# ----- Main neighbor_batch_predict implementation (uses induced subgraph) -----
def neighbor_batch_predict(model, full_data, target_node_idx, neighbors_per_layer, device="cpu", seed=None):
    """
    model: GraphSAGE model
    full_data: torch_geometric.data.Data with attributes x (NxD) and edge_index (2xE)
    target_node_idx: array-like of node indices in the full graph to predict
    neighbors_per_layer: list of ints for neighbor counts per layer (outer->inner)
    Returns:
      preds: np.array of predicted class indices (len = len(target_node_idx))
      probs: np.array of softmax probability vectors (len x num_classes)
    """
    model.eval()
    x_all_np = full_data.x.cpu().numpy()
    edge_index_np = full_data.edge_index.cpu().numpy().astype(int)
    n_all = x_all_np.shape[0]
    adj = build_adj_list(edge_index_np, num_nodes=n_all)

    preds = []
    probs_list = []
    for t in np.array(target_node_idx).flatten():
        # expand nodes for this target
        nodes, old_to_new = build_subgraph_nodes([int(t)], adj, neighbors_per_layer, seed=seed)
        x_sub, edge_index_sub = induced_subgraph(nodes, old_to_new, edge_index_np, x_all_np)
        # map target index to new index
        target_new_idx = old_to_new[int(t)]
        # move to device
        x_sub = x_sub.to(device)
        edge_index_sub = edge_index_sub.to(device)
        with torch.no_grad():
            out = model(x_sub, edge_index_sub)  # shape: (n_sub, out_dim)
            logits = out[target_new_idx].cpu()
            probs = F.softmax(logits, dim=0).numpy()
            pred_id = int(np.argmax(probs))
        preds.append(pred_id)
        probs_list.append(probs)
    return np.array(preds, dtype=int), np.vstack(probs_list)

# ------------------- Script begins here -------------------
def main():
    # basic file checks
    for path in [PLAIN_PATH, MODEL_PATH, VECTORIZER_PATH]:
        if not os.path.exists(path):
            print(f"Required file not found: {path}")
            print("Make sure your working directory contains sampled_data/ with the required files.")
            sys.exit(1)

    # load vectorizer and plain graph data
    vec = joblib.load(VECTORIZER_PATH)
    plain = torch.load(PLAIN_PATH, map_location="cpu")
    if "x" not in plain or "edge_index" not in plain:
        raise ValueError("graph_data_plain.pt must be a dict with keys 'x' and 'edge_index'")

    existing_x = plain["x"].cpu().numpy().astype(np.float32)  # NxD
    edge_index_old = plain["edge_index"].cpu().numpy().astype(int)  # 2xE

    in_channels = existing_x.shape[1]
    num_classes = len(DBPEDIA_14_NAMES)

    # construct model and load state
    model = GraphSAGE(in_channels, HIDDEN, num_classes, num_layers=NUM_LAYERS, dropout=DROPOUT)
    state = torch.load(MODEL_PATH, map_location="cpu")
    # if state is a dict with keys 'model_state_dict' etc, try to find the right key
    if isinstance(state, dict) and ('state_dict' in state and isinstance(state['state_dict'], dict)):
        sd = state['state_dict']
    elif isinstance(state, dict) and ('model_state_dict' in state and isinstance(state['model_state_dict'], dict)):
        sd = state['model_state_dict']
    else:
        sd = state
    model.load_state_dict(sd)
    model.to(DEVICE)
    model.eval()

    # optional label mapping
    label_map = None
    if os.path.exists(LABEL_MAP_PATH):
        try:
            label_map = joblib.load(LABEL_MAP_PATH)
        except Exception:
            label_map = None

    # convert input text to TF-IDF vector
    x_new = vec.transform([INPUT_TEXT]).toarray().astype(np.float32)

    # find K nearest TF-IDF neighbors to attach the new node
    from sklearn.neighbors import NearestNeighbors
    knn = NearestNeighbors(n_neighbors=min(K_TFIDF, existing_x.shape[0]), metric='cosine', n_jobs=-1)
    knn.fit(existing_x)
    distances, neigh = knn.kneighbors(x_new, return_distance=True)  # neigh is shape (1,K)
    neigh = neigh[0].astype(int)

    # create new edges connecting new node to these neighbors (bidirectional)
    n_old = existing_x.shape[0]
    new_idx = n_old
    u = []
    v = []
    for nb in neigh:
        u.append(new_idx); v.append(int(nb))
        u.append(int(nb)); v.append(new_idx)

    # Build the augmented feature matrix and edge_index
    all_x = np.vstack([existing_x, x_new]).astype(np.float32)
    u_all = np.concatenate([edge_index_old[0], np.array(u, dtype=int)])
    v_all = np.concatenate([edge_index_old[1], np.array(v, dtype=int)])
    edge_index_new = np.vstack([u_all, v_all]).astype(np.int64)

    temp_data = Data(x=torch.from_numpy(all_x), edge_index=torch.from_numpy(edge_index_new))

    # run neighbor-batch (subgraph) predict on the single new node
    preds, probs = neighbor_batch_predict(
        model,
        temp_data,
        target_node_idx=np.array([new_idx], dtype=int),
        neighbors_per_layer=NEIGHBORS_PER_LAYER,
        device=DEVICE,
        seed=42
    )

    pred_id = int(preds[0])
    prob_vector = probs[0] if probs.size else None

    # final label name (prefer label_map if provided)
    if label_map is not None:
        try:
            pred_name = label_map[pred_id]
        except Exception:
            pred_name = DBPEDIA_14_NAMES[pred_id] if 0 <= pred_id < len(DBPEDIA_14_NAMES) else "UNKNOWN"
    else:
        pred_name = DBPEDIA_14_NAMES[pred_id] if 0 <= pred_id < len(DBPEDIA_14_NAMES) else "UNKNOWN"

    print("Prediction (DBpedia-14):")
    print("  class_id   :", pred_id)
    print("  class_name :", pred_name)
    if prob_vector is not None:
        # print probability for each DBpedia class label
        for i, p in enumerate(prob_vector):
            label = (label_map[i] if (label_map is not None and i in label_map) else
                     (DBPEDIA_14_NAMES[i] if i < len(DBPEDIA_14_NAMES) else str(i)))
            print(f"    {i:02d} {label:25s} : {p:.4f}")

if __name__ == "__main__":
    main()


Prediction (DBpedia-14):
  class_id   : 7
  class_name : NaturalPlace
    00 Company                   : 0.0001
    01 EducationalInstitution    : 0.0001
    02 Artist                    : 0.0000
    03 Athlete                   : 0.0000
    04 OfficeHolder              : 0.0000
    05 MeanOfTransportation      : 0.0001
    06 Building                  : 0.0002
    07 NaturalPlace              : 0.9987
    08 Village                   : 0.0001
    09 Animal                    : 0.0003
    10 Plant                     : 0.0001
    11 Album                     : 0.0001
    12 Film                      : 0.0001
    13 WrittenWork               : 0.0001


: 